## The Core Insight: Memory is the Bottleneck
Most ML kernels are memory-bound, not compute-bound. The GPU can do math far faster than it can fetch data from HBM. This is why:
- **Kernel fusion matters**: fewer round-trips to HBM
- **Tiling matters**: load a block of data into SRAM, do all your math, write back once
- **Triton's value proposition**: it makes tiling easy to express, compiler handles the rest

## Exercise: Profile a Simple PyTorch Op

In [1]:
import torch
import torch.utils.benchmark as benchmark

x = torch.randn(4096, 4096, device='cuda')

# Two separate ops 
# even though you did not explicitly write two kernels, PyTorch’s eager execution model treats mul and add as separate ops, each commonly backed by its own CUDA kernel.
#(2 HBM round-trips)
def two_ops(x):
    return (x * 2) + 1

# Measure
t = benchmark.Timer(stmt='two_ops(x)', globals={'two_ops': two_ops, 'x': x})
print(t.timeit(100))

two_ops(x)
  1.13 ms
  1 measurement, 100 runs , 1 thread


**Question to answer**: How much time is spent on memory transfer vs. actual compute? 